# Session 04 · Retrieve Data from Multiple Tables

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - INNER JOIN: students with their batch and course

In [2]:
from campus_db import connect, show
con = connect()
show(con, """SELECT s.name, s.batch_id, b.course, b.timing
FROM students AS s
INNER JOIN batches AS b ON b.batch_id = s.batch_id
ORDER BY b.course, s.name""")

name          | batch_id | course          | timing 
--------------+----------+-----------------+--------
Omar Farooq   | DS-2601  | Data Science    | Weekend
Zara Hussain  | DS-2601  | Data Science    | Weekend
Ali Raza      | AI-2601  | Diploma in AI   | Morning
Ayesha Khan   | AI-2601  | Diploma in AI   | Morning
Bilal Ahmed   | AI-2601  | Diploma in AI   | Morning
Fatima Noor   | AI-2602  | Diploma in AI   | Evening
Hira Siddiqui | AI-2602  | Diploma in AI   | Evening
Hamza Malik   | WD-2601  | Web Development | Evening
Mariam Javed  | WD-2601  | Web Development | Evening
(9 rows)



[('Omar Farooq', 'DS-2601', 'Data Science', 'Weekend'),
 ('Zara Hussain', 'DS-2601', 'Data Science', 'Weekend'),
 ('Ali Raza', 'AI-2601', 'Diploma in AI', 'Morning'),
 ('Ayesha Khan', 'AI-2601', 'Diploma in AI', 'Morning'),
 ('Bilal Ahmed', 'AI-2601', 'Diploma in AI', 'Morning'),
 ('Fatima Noor', 'AI-2602', 'Diploma in AI', 'Evening'),
 ('Hira Siddiqui', 'AI-2602', 'Diploma in AI', 'Evening'),
 ('Hamza Malik', 'WD-2601', 'Web Development', 'Evening'),
 ('Mariam Javed', 'WD-2601', 'Web Development', 'Evening')]

## Example 2 - joining three tables

In [3]:
show(con, """SELECT s.name AS student, b.course, f.name AS teacher
FROM students s
JOIN batches  b ON b.batch_id   = s.batch_id
JOIN faculty  f ON f.faculty_id = b.faculty_id
ORDER BY teacher, student""")

student       | course          | teacher     
--------------+-----------------+-------------
Hamza Malik   | Web Development | Kamran Ali  
Mariam Javed  | Web Development | Kamran Ali  
Omar Farooq   | Data Science    | Mehwish Raza
Zara Hussain  | Data Science    | Mehwish Raza
Ali Raza      | Diploma in AI   | Sana Iqbal  
Ayesha Khan   | Diploma in AI   | Sana Iqbal  
Bilal Ahmed   | Diploma in AI   | Sana Iqbal  
Fatima Noor   | Diploma in AI   | Usman Tariq 
Hira Siddiqui | Diploma in AI   | Usman Tariq 
(9 rows)



[('Hamza Malik', 'Web Development', 'Kamran Ali'),
 ('Mariam Javed', 'Web Development', 'Kamran Ali'),
 ('Omar Farooq', 'Data Science', 'Mehwish Raza'),
 ('Zara Hussain', 'Data Science', 'Mehwish Raza'),
 ('Ali Raza', 'Diploma in AI', 'Sana Iqbal'),
 ('Ayesha Khan', 'Diploma in AI', 'Sana Iqbal'),
 ('Bilal Ahmed', 'Diploma in AI', 'Sana Iqbal'),
 ('Fatima Noor', 'Diploma in AI', 'Usman Tariq'),
 ('Hira Siddiqui', 'Diploma in AI', 'Usman Tariq')]

## Example 3 - LEFT JOIN keeps rows with no match (find who is missing)

In [4]:
show(con, """SELECT s.name, COALESCE(b.course, '-- not placed --') AS course
FROM students s LEFT JOIN batches b ON b.batch_id = s.batch_id""")
show(con, """SELECT b.batch_id, b.course
FROM batches b LEFT JOIN students s ON s.batch_id = b.batch_id
WHERE s.student_id IS NULL""")                     # batches with no students yet

name          | course          
--------------+-----------------
Ayesha Khan   | Diploma in AI   
Bilal Ahmed   | Diploma in AI   
Hira Siddiqui | Diploma in AI   
Omar Farooq   | Data Science    
Zara Hussain  | Data Science    
Hamza Malik   | Web Development 
Fatima Noor   | Diploma in AI   
Ali Raza      | Diploma in AI   
Mariam Javed  | Web Development 
Saad Qureshi  | -- not placed --
(10 rows)

batch_id | course          
---------+-----------------
ML-2701  | Machine Learning
(1 rows)



[('ML-2701', 'Machine Learning')]

## Example 4 - RIGHT and FULL OUTER JOIN

In [5]:
show(con, """SELECT f.name AS teacher, b.batch_id
FROM batches b RIGHT JOIN faculty f ON f.faculty_id = b.faculty_id""")
show(con, """SELECT f.name AS teacher, b.batch_id
FROM faculty f FULL OUTER JOIN batches b ON b.faculty_id = f.faculty_id
ORDER BY teacher""")

teacher      | batch_id
-------------+---------
Sana Iqbal   | AI-2601 
Usman Tariq  | AI-2602 
Mehwish Raza | DS-2601 
Kamran Ali   | WD-2601 
Nida Farooq  | None    
(5 rows)

teacher      | batch_id
-------------+---------
None         | ML-2701 
Kamran Ali   | WD-2601 
Mehwish Raza | DS-2601 
Nida Farooq  | None    
Sana Iqbal   | AI-2601 
Usman Tariq  | AI-2602 
(6 rows)



[(None, 'ML-2701'),
 ('Kamran Ali', 'WD-2601'),
 ('Mehwish Raza', 'DS-2601'),
 ('Nida Farooq', None),
 ('Sana Iqbal', 'AI-2601'),
 ('Usman Tariq', 'AI-2602')]

## Example 5 - self join and CROSS JOIN

In [6]:
show(con, """SELECT a.name AS student, b.name AS classmate, a.batch_id
FROM students a JOIN students b ON a.batch_id = b.batch_id AND a.student_id < b.student_id
ORDER BY a.batch_id""")
show(con, """SELECT f.name, t.slot FROM faculty f
CROSS JOIN (SELECT 'Morning' AS slot UNION ALL SELECT 'Evening') AS t
WHERE f.subject IN ('Python', 'SQL')""")

student       | classmate    | batch_id
--------------+--------------+---------
Ayesha Khan   | Ali Raza     | AI-2601 
Ayesha Khan   | Bilal Ahmed  | AI-2601 
Bilal Ahmed   | Ali Raza     | AI-2601 
Hira Siddiqui | Fatima Noor  | AI-2602 
Omar Farooq   | Zara Hussain | DS-2601 
Hamza Malik   | Mariam Javed | WD-2601 
(6 rows)

name        | slot   
------------+--------
Sana Iqbal  | Morning
Sana Iqbal  | Evening
Usman Tariq | Morning
Usman Tariq | Evening
(4 rows)



[('Sana Iqbal', 'Morning'),
 ('Sana Iqbal', 'Evening'),
 ('Usman Tariq', 'Morning'),
 ('Usman Tariq', 'Evening')]

## Example 6 - set operators: UNION, UNION ALL, INTERSECT, EXCEPT

In [7]:
show(con, """SELECT name, 'student' AS role FROM students WHERE city = 'Lahore'
UNION
SELECT name, 'faculty' FROM faculty WHERE subject = 'SQL'""")
show(con, "SELECT student_id FROM marks WHERE subject = 'Python' INTERSECT SELECT student_id FROM marks WHERE subject = 'SQL'")
show(con, "SELECT student_id FROM students EXCEPT SELECT student_id FROM marks")      # no marks yet

name        | role   
------------+--------
Bilal Ahmed | student
Fatima Noor | student
Usman Tariq | faculty
(3 rows)

student_id
----------
101       
102       
103       
107       
108       
(5 rows)

student_id
----------
110       
(1 rows)



[(110,)]

## Example 7 - subqueries: scalar, IN, EXISTS and correlated

In [8]:
show(con, "SELECT student_id, subject, marks FROM marks WHERE marks > (SELECT AVG(marks) FROM marks)")
show(con, "SELECT name FROM students WHERE batch_id IN (SELECT batch_id FROM batches WHERE timing = 'Evening')")
show(con, "SELECT f.name FROM faculty f WHERE NOT EXISTS (SELECT 1 FROM batches b WHERE b.faculty_id = f.faculty_id)")
show(con, """SELECT m.student_id, m.subject, m.marks FROM marks m
WHERE m.marks = (SELECT MAX(m2.marks) FROM marks m2 WHERE m2.subject = m.subject)""")  # topper per subject

student_id | subject    | marks
-----------+------------+------
101        | Python     | 88   
101        | SQL        | 79   
103        | Python     | 92   
103        | SQL        | 95   
105        | Statistics | 83   
105        | Python     | 77   
107        | Python     | 74   
107        | SQL        | 81   
109        | Web        | 90   
(9 rows)

name         
-------------
Hira Siddiqui
Hamza Malik  
Fatima Noor  
Mariam Javed 
(4 rows)

name       
-----------
Nida Farooq
(1 rows)

student_id | subject    | marks
-----------+------------+------
103        | Python     | 92   
103        | SQL        | 95   
105        | Statistics | 83   
109        | Web        | 90   
(4 rows)



[(103, 'Python', 92),
 (103, 'SQL', 95),
 (105, 'Statistics', 83),
 (109, 'Web', 90)]